# TurnRelevancyMetric

## What it measures

In a multi-turn conversation, whether each assistant turn is relevant **given everything
said before it**. The judge slides a window over the transcript and asks, for each
assistant response, whether it addresses the user's message in the context of the
conversation so far. The score is the proportion of relevant assistant turns.

Single-turn Answer Relevancy cannot see this. A follow-up like "Which jurisdictions are on
that list?" is unanswerable in isolation - "that list" only means something because of the
previous turn. An assistant that answers it correctly is doing conversational work that a
per-turn metric would either miss or punish.

## When it is useful

On any stateful assistant where the user is expected to ask follow-ups. It is the metric
that catches conversation memory being dropped, truncated too aggressively, or - the
subtler failure - carried so heavily that the assistant keeps answering the *first*
question instead of the current one.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `ConversationalTestCase` (multi-turn) |
| `turns` | yes - a list of `Turn(role, content)`, alternating user and assistant |

`window_size` defaults to `10`, meaning each assistant turn is judged against up to ten
preceding turns.

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
# The contract these notebooks were written against. The application may
# serve a HIGHER minor version: a MINOR bump is additive by its own
# contract policy (1.0.0 -> 1.1.0 added HealthResponse.build_version and
# changed nothing else), so treating it as a mismatch would turn this
# guard into noise on every single call. Only a MAJOR change, or an
# application older than these notebooks, is a problem.
EXPECTED_SCHEMA_VERSION = "1.0.0"


def contract_version(value):
    """(major, minor) from a MAJOR.MINOR.PATCH string, or None."""
    try:
        parts = value.split("+")[0].split(".")
        return int(parts[0]), int(parts[1])
    except (AttributeError, IndexError, ValueError):
        return None

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    served_version = contract_version(served) if served else None
    expected_version = contract_version(EXPECTED_SCHEMA_VERSION)
    if served_version and served_version[0] != expected_version[0]:
        raise ApiError(
            f"The application serves contract version {served}; these notebooks were "
            f"written against {EXPECTED_SCHEMA_VERSION}. A MAJOR change means fields "
            f"may have been removed or retyped - re-derive the goldens against the "
            f"new contract rather than scoring against one they do not match."
        )
    if served_version and served_version[1] < expected_version[1]:
        print(f"WARNING: application reports contract version {served}, older than "
              f"the {EXPECTED_SCHEMA_VERSION} these notebooks were written against. "
              f"Fields the goldens rely on may not exist yet.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoints exercised

| Endpoint | Role here |
|---|---|
| `DELETE /api/cases/{case_id}/conversation` | Clears the thread so the transcript starts from a known state |
| `POST /api/rag/query` (three times, `include_history` left at its default `true`) | Produces the conversation |
| `GET /api/cases/{case_id}/conversation` | The authoritative transcript, read back independently |

This is the one metric in the suite whose multi-turn structure is **genuinely the
application's own**, not assembled by the harness. `POST /api/rag/query` is case-scoped and
stateful: every case has at most one conversation thread, created on the first question and
appended to on every later one.

The transcript is read back from `GET /api/cases/{case_id}/conversation` rather than
stitched together from the three responses. That matters: it means the conversation being
scored is the one the *application* recorded, so a turn the application dropped or
reordered shows up as a scoring input rather than being papered over by the harness's own
bookkeeping.

The questions are deliberately built to depend on each other - turn 2 refers to "that
list", turn 3 to "this case" - because a conversation of independent questions would score
identically with memory switched off, and would test nothing.

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

In [ ]:
# --------------------------------------------------------------------------
# The exact requests.
#
# include_history is left at its default (true) - this notebook is testing
# conversation handling, so history is the subject, not a confound.
#
# The thread is cleared first so the transcript starts from a known state.
# DELETE is used rather than dev/reset because it is scoped to one case and
# destroys nothing else.
# --------------------------------------------------------------------------
CASE_ID = SCENARIOS["s2"]["case_id"]      # "High-risk jurisdiction transfer"

cleared = api("DELETE", f"/api/cases/{CASE_ID}/conversation")
print("DELETE", f"{API_BASE}/api/cases/{CASE_ID}/conversation", "->", json.dumps(cleared))
print()

QUESTIONS = [
    # Turn 1 - self-contained.
    "What does policy require before releasing an outbound wire to a high-risk jurisdiction?",
    # Turn 2 - "that list" is only resolvable from turn 1.
    "Which jurisdictions are on that list?",
    # Turn 3 - "this case" depends on the case scope and on turns 1 and 2.
    "Does the transaction on this case fall under those requirements?",
]

print("POST", f"{API_BASE}/api/rag/query", "(x3, same case, history enabled)")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
for n, question in enumerate(QUESTIONS, start=1):
    print(f"body[{n}]:", json.dumps({"question": question, "case_id": CASE_ID,
                                     "top_k": 6}))

In [ ]:
# --------------------------------------------------------------------------
# The raw responses.
#
# retrieval_query is the interesting field: it is the string the retriever
# actually embedded. On turn 1 it equals `question`; on later turns it is
# history-expanded, which is how a pronoun-only follow-up still retrieves the
# right chunks. Scoring retrieval against `question` alone would score text the
# retriever never saw.
# --------------------------------------------------------------------------
responses = []
for n, question in enumerate(QUESTIONS, start=1):
    response = api("POST", "/api/rag/query",
                   json_body={"question": question, "case_id": CASE_ID, "top_k": 6})
    responses.append(response)
    print(f"--- turn {n} ---")
    print(f"  run_id             : {response['run_id']}")
    print(f"  conversation_id    : {response['conversation_id']}")
    print(f"  history_turns_used : {response['history_turns_used']}")
    print(f"  question           : {response['question']}")
    print(f"  retrieval_query    : {response['retrieval_query']}")
    print(f"  grounding          : {response['grounding']}")
    print(f"  answer             : {response['answer'][:220]}...")
    print()

conversation_ids = {r["conversation_id"] for r in responses}
print(f"all three turns joined conversation_id(s): {conversation_ids}")
if len(conversation_ids) != 1:
    raise RuntimeError(
        f"Expected all three turns on one conversation thread, got {conversation_ids}. "
        f"Conversation memory is case-scoped; a differing id means the turns were not "
        f"threaded and there is no multi-turn behaviour to score."
    )
if responses[1]["history_turns_used"] == 0:
    raise RuntimeError(
        "Turn 2 reports history_turns_used = 0, so no prior turn was fed back in. The "
        "conversation is not actually multi-turn and TurnRelevancyMetric would be "
        "scoring three unrelated exchanges."
    )

In [ ]:
# --------------------------------------------------------------------------
# Read the authoritative transcript back from the application.
# --------------------------------------------------------------------------
conversation = api("GET", f"/api/cases/{CASE_ID}/conversation")

print(f"case_id         : {conversation['case_id']}")
print(f"conversation_id : {conversation['conversation_id']}")
print(f"turn_count      : {conversation['turn_count']}")
print()
for turn in conversation["turns"]:
    print(f"--- turn_index {turn['turn_index']} (run_id {turn['run_id']}) ---")
    print("  Q:", textwrap.fill(turn["question"], width=92, subsequent_indent="     "))
    print("  A:", textwrap.fill(turn["answer"][:400], width=92, subsequent_indent="     "))
    print()

if conversation["turn_count"] < len(QUESTIONS):
    raise RuntimeError(
        f"The application recorded {conversation['turn_count']} turns but "
        f"{len(QUESTIONS)} questions were asked. Scoring an incomplete transcript would "
        f"misattribute a persistence bug to the assistant's relevance."
    )

## Mapping the API response onto DeepEval fields

| DeepEval field | API field | Note |
|---|---|---|
| `turns[i]` with `role="user"` | `turns[].question` | From the transcript, in `turn_index` order |
| `turns[i]` with `role="assistant"` | `turns[].answer` | |
| `ConversationalTestCase.turns` | the flattened alternating list | |

Each recorded turn becomes **two** DeepEval `Turn` objects, because the application stores a
question and its answer as one row while DeepEval models a conversation as a flat sequence
of messages.

`retrieval_context` is deliberately left off the turns. This metric judges relevance of the
response to the conversation, and attaching retrieval would invite the judge to reason about
grounding instead - a different question, covered by the RAG metrics.

## No golden is derived

Turn Relevancy is reference-free. What *is* derived from the API is the transcript itself,
read back from `GET /api/cases/{case_id}/conversation` rather than reconstructed locally,
so the conversation scored is the application's own record. The assertions in the cells
above - one shared `conversation_id`, non-zero `history_turns_used`, and a complete
transcript - are what make a low score attributable to relevance rather than to the
conversation never having been threaded in the first place.

In [ ]:
# --------------------------------------------------------------------------
# Build the test case and print the multi-turn history explicitly.
# --------------------------------------------------------------------------
from deepeval.test_case import ConversationalTestCase, Turn

turns = []
for recorded in sorted(conversation["turns"], key=lambda t: t["turn_index"]):
    turns.append(Turn(role="user", content=recorded["question"]))
    turns.append(Turn(role="assistant", content=recorded["answer"]))

test_case = ConversationalTestCase(turns=turns)

print(f"MULTI-TURN HISTORY: {len(test_case.turns)} turns "
      f"({len(test_case.turns) // 2} exchanges)")
print()
for n, turn in enumerate(test_case.turns):
    label = "USER INPUT     " if turn.role == "user" else "ACTUAL OUTPUT  "
    print(f"[{n}] {label} {textwrap.fill(turn.content[:300], width=76, subsequent_indent=' ' * 20)}")
print()
print("EXPECTED OUTPUT (golden) : not used by this metric")
print("RETRIEVAL CONTEXT        : deliberately not attached - see the mapping section")

## Judge and threshold

- **Judge model**: `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `TurnRelevancyMetric`.
- **`window_size=10`**: DeepEval's default, stated explicitly. It is the number of
  preceding turns each assistant response is judged against. With a three-exchange
  conversation the whole transcript fits inside one window, so no turn is judged with part
  of its context hidden.

The default threshold is kept. The score is the proportion of relevant assistant turns, so
with three exchanges the granularity is one third - `0.5` means at least two of three
responses stayed on topic. Note that the application's own memory window is six turns, so a
conversation longer than that would begin dropping context on the application side while
DeepEval's window still spanned it; keeping the conversation short avoids conflating the
two limits.

In [ ]:
from deepeval.metrics import TurnRelevancyMetric

metric = TurnRelevancyMetric(
    threshold=0.5,          # DeepEval's documented default
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
    window_size=10,         # DeepEval's default sliding window
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

## Limitations in a black-box acceptance test

1. **Relevance is not grounding.** A perfectly on-topic follow-up can still be wrong. This
   metric cannot see the retrieved evidence, and does not try to.
2. **Three exchanges is a short conversation.** The application drops turns older than six
   from the prompt (and enforces a token budget as well). A transcript short enough to sit
   inside both windows never exercises that truncation, so this notebook does not test what
   happens when memory starts being discarded - which is where the interesting failures
   live.
3. **The retrieval query is expanded from history, and the metric never sees it.**
   `retrieval_query` is printed above precisely because a follow-up can be answered
   relevantly from wrongly-expanded retrieval, or irrelevantly from correct retrieval, and
   the score alone cannot distinguish them.
4. **Conversation state is shared and durable.** The thread lives on the case, not on this
   run. Another client asking questions about the same case interleaves with this
   transcript. The `DELETE` at the start bounds that risk but does not eliminate it in a
   parallel suite - conversation-based tests should not run concurrently against one case.
5. **Only the RAG surface is conversational.** `POST /api/cases/{id}/investigate` has no
   conversation state at all, so this metric says nothing about the agentic path. That
   asymmetry is why the two MCP notebooks have to assemble their own transcripts.